# Unit 2: ML Data Preparation

**Date:** Wednesday, December 17, 2025

**Instructor:** Abishek Ganesh

---

## Today's Goal
Transform our EDA-cleaned data into a **Machine Learning-ready format**.

### The Bridge: EDA → ML
Yesterday we explored and cleaned our data. Today we prepare it for modeling.

| EDA Output | ML Input Requirement |
|------------|---------------------|
| Mixed data types (text + numbers) | **All numeric** |
| Values in different ranges (age: 18-64, charges: 1K-60K) | **Scaled to similar ranges** |
| Categories like "yes"/"no", "male"/"female" | **Encoded as 0/1** |
| ID columns, irrelevant features | **Dropped** |

---

## The 4-Step ML Data Prep Process

1. **Drop** unnecessary columns
2. **Encode** categorical variables → numbers
3. **Scale** numerical features → similar ranges
4. **Split** into training and test sets

---

# PART 1: Instructor-Led Walkthrough
## Dataset: Medical Insurance Costs

We'll continue with the same dataset from our EDA practice.

## 1.1 Setup & Data Loading

In [1]:
# Import libraries
import pandas as pd
import numpy as np

# Sklearn preprocessing tools
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split

print("Libraries loaded!")

Libraries loaded!


In [3]:
# Load the data (same dataset from EDA practice)
df = pd.read_csv('medical_cost.csv')

# Quick clean from EDA - remove duplicates
df = df.drop_duplicates()

print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (1337, 7)


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [4]:
# Let's see what we're working with
print("Current data types:")
print(df.dtypes)
print("\n" + "="*50)
print("\nCategorical columns (need encoding):")
print(df.select_dtypes(include=['object']).columns.tolist())
print("\nNumerical columns:")
print(df.select_dtypes(include=['number']).columns.tolist())

Current data types:
age           int64
sex          object
bmi         float64
children      int64
smoker       object
region       object
charges     float64
dtype: object


Categorical columns (need encoding):
['sex', 'smoker', 'region']

Numerical columns:
['age', 'bmi', 'children', 'charges']


### Current State of Our Data

**Problem:** We have 3 text/categorical columns that ML algorithms can't understand:
- `sex`: "male" or "female"
- `smoker`: "yes" or "no"
- `region`: "southwest", "southeast", "northwest", "northeast"

**Also:** Our numerical values are on very different scales:
- `age`: 18-64
- `bmi`: ~16-53
- `children`: 0-5
- `charges`: ~1,100-63,000

Let's fix both issues!

---

## 1.2 Step 1: Drop Unnecessary Columns

**What to drop:**
- ID columns (just row identifiers, no predictive value)
- Columns that "leak" the target (contain information about the answer)
- Highly redundant columns

**This dataset:** No ID column, so nothing to drop here. But the stroke dataset has one!

In [ ]:
# This dataset is clean - no columns to drop
# But here's how you would do it:

# df = df.drop(columns=['id'])  # Drop single column
# df = df.drop(columns=['id', 'name', 'timestamp'])  # Drop multiple

print("Columns in our dataset:")
print(df.columns.tolist())
print("\nNo unnecessary columns to drop in this dataset!")

Columns in our dataset:
['age', 'sex', 'bmi', 'children', 'smoker', 'region', 'charges']

No unnecessary columns to drop in this dataset!


---

## 1.3 Step 2: Encode Categorical Variables

We need to convert text categories to numbers. Two main approaches:

### Option A: Label Encoding (for binary categories)
- Converts categories to 0, 1, 2, etc.
- Best for: Binary (2 options) or ordinal (has order) categories
- Example: "yes"/"no" → 1/0

### Option B: One-Hot Encoding (for multiple categories)
- Creates a new column for each category (0 or 1)
- Best for: Nominal categories (no natural order)
- Example: "region" → 4 new columns

### 1.3.1 Label Encoding for Binary Columns

In [6]:
# Before encoding - let's see our categorical values
print("BEFORE ENCODING:")
print("\nsex values:", df['sex'].unique())
print("smoker values:", df['smoker'].unique())
print("region values:", df['region'].unique())

BEFORE ENCODING:

sex values: ['female' 'male']
smoker values: ['yes' 'no']
region values: ['southwest' 'southeast' 'northwest' 'northeast']


In [7]:
# Create a copy to work with
df_encoded = df.copy()

# Label encode binary columns: sex and smoker
le = LabelEncoder()

# Encode 'sex': female=0, male=1
df_encoded['sex'] = le.fit_transform(df_encoded['sex'])
print("sex encoding:", dict(zip(le.classes_, range(len(le.classes_)))))

# Encode 'smoker': no=0, yes=1
df_encoded['smoker'] = le.fit_transform(df_encoded['smoker'])
print("smoker encoding:", dict(zip(le.classes_, range(len(le.classes_)))))

sex encoding: {'female': 0, 'male': 1}
smoker encoding: {'no': 0, 'yes': 1}


In [8]:
# Check the result
df_encoded[['sex', 'smoker']].head(10)

,sex,smoker
0,0,1
1,1,0
2,1,0
3,1,0
4,1,0
5,0,0
6,0,0
7,0,0
8,1,0
9,0,0


### 1.3.2 One-Hot Encoding for Multi-Category Columns

In [9]:
# One-hot encode 'region' (4 categories)
# pd.get_dummies() is the easiest way

df_encoded = pd.get_dummies(df_encoded, columns=['region'], prefix='region')

print("New columns after one-hot encoding:")
print(df_encoded.columns.tolist())

New columns after one-hot encoding:
['age', 'sex', 'bmi', 'children', 'smoker', 'charges', 'region_northeast', 'region_northwest', 'region_southeast', 'region_southwest']


In [10]:
# See the one-hot encoded region columns
df_encoded[['region_northeast', 'region_northwest', 'region_southeast', 'region_southwest']].head(10)

,region_northeast,region_northwest,region_southeast,region_southwest
0,False,False,False,True
1,False,False,True,False
2,False,False,True,False
3,False,True,False,False
4,False,True,False,False
5,False,False,True,False
6,False,False,True,False
7,False,True,False,False
8,True,False,False,False
9,False,True,False,False


In [11]:
# Verify: All columns are now numeric!
print("Data types after encoding:")
print(df_encoded.dtypes)

Data types after encoding:
age                   int64
sex                   int64
bmi                 float64
children              int64
smoker                int64
charges             float64
region_northeast       bool
region_northwest       bool
region_southeast       bool
region_southwest       bool
dtype: object


### Encoding Complete!

| Original Column | Encoding Method | Result |
|-----------------|-----------------|--------|
| sex | Label Encoding | 0 (female), 1 (male) |
| smoker | Label Encoding | 0 (no), 1 (yes) |
| region | One-Hot Encoding | 4 new columns (0 or 1 each) |

---

## 1.4 Step 3: Scale Numerical Features

### Why Scale?

Many ML algorithms are sensitive to the **magnitude** of values:
- `age` ranges from 18-64 (range of ~46)
- `charges` ranges from 1,100-63,000 (range of ~62,000)

Without scaling, the algorithm might think `charges` is 1000x more important just because its numbers are bigger!

---

### ⚠️ Important: When to Scale (Data Leakage Warning)

**In this demo**, we're scaling our data before splitting into train/test sets. This is fine for learning the concepts.

**In real ML projects**, you should:
1. **Split first** → Get your train and test sets
2. **Fit the scaler on training data only** → `scaler.fit(X_train)`
3. **Transform both sets** → `scaler.transform(X_train)` and `scaler.transform(X_test)`

**Why does this matter?**

If you scale before splitting, information from your test set "leaks" into your training process. The scaler learns the mean and standard deviation from ALL the data (including test data), which gives your model an unfair advantage.

This is called **data leakage** - your model gets hints about the test data it's not supposed to see!

```python
# THE RIGHT WAY (in real projects):
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # Fit on training only
X_test_scaled = scaler.transform(X_test)        # Transform test (no fit!)
```

---

### Two Main Scaling Methods

| Method | What It Does | Output Range | Best For |
|--------|--------------|--------------|----------|
| **StandardScaler** | Centers around mean, scales by std dev | Roughly **-3 to +3** | Most ML algorithms (linear regression, neural networks, SVM) |
| **MinMaxScaler** | Scales to a fixed range | Exactly **0 to 1** | When you need bounded values (image data, neural networks with sigmoid) |

---

### StandardScaler (Standardization)
Transforms data to have:
- **Mean = 0**
- **Standard deviation = 1**

Formula: `z = (x - mean) / std`

**Use when:** You want values centered around zero. Works well with most algorithms.

### MinMaxScaler (Normalization)
Transforms data to a fixed range:
- **Minimum = 0**
- **Maximum = 1**

Formula: `x_scaled = (x - min) / (max - min)`

**Use when:** You need values strictly between 0 and 1, or your data doesn't have many outliers.

---

### Quick Recommendation Guide

| Situation | Recommended Scaler |
|-----------|-------------------|
| General ML models (regression, classification) | **StandardScaler** |
| Data has outliers | **StandardScaler** (more robust) |
| Neural networks | Either works, StandardScaler common |
| Need values exactly 0-1 | **MinMaxScaler** |
| Image pixel data | **MinMaxScaler** |
| Data is uniformly distributed | **MinMaxScaler** |

**When in doubt, use StandardScaler** - it's the most common choice and works well in most situations.

In [12]:
# First, separate features (X) from target (y)
# Target = what we want to predict (charges)

X = df_encoded.drop(columns=['charges'])  # Features (everything except charges)
y = df_encoded['charges']                  # Target (what we predict)

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFeature columns: {X.columns.tolist()}")

Features shape: (1337, 9)
Target shape: (1337,)

Feature columns: ['age', 'sex', 'bmi', 'children', 'smoker', 'region_northeast', 'region_northwest', 'region_southeast', 'region_southwest']


In [13]:
# BEFORE scaling - look at the value ranges
print("BEFORE SCALING:")
print(X[['age', 'bmi', 'children']].describe().round(2))

BEFORE SCALING:
           age      bmi  children
count  1337.00  1337.00   1337.00
mean     39.22    30.66      1.10
std      14.04     6.10      1.21
min      18.00    15.96      0.00
25%      27.00    26.29      0.00
50%      39.00    30.40      1.00
75%      51.00    34.70      2.00
max      64.00    53.13      5.00


In [14]:
# Apply StandardScaler to numerical columns
scaler = StandardScaler()

# Columns to scale (the original numerical ones, not the encoded ones)
cols_to_scale = ['age', 'bmi', 'children']

# Fit and transform
X_scaled = X.copy()
X_scaled[cols_to_scale] = scaler.fit_transform(X[cols_to_scale])

print("Scaling complete!")

Scaling complete!


In [15]:
# AFTER scaling - values are now centered around 0
print("AFTER SCALING:")
print(X_scaled[['age', 'bmi', 'children']].describe().round(2))

AFTER SCALING:
           age      bmi  children
count  1337.00  1337.00   1337.00
mean     -0.00    -0.00      0.00
std       1.00     1.00      1.00
min      -1.51    -2.41     -0.91
25%      -0.87    -0.72     -0.91
50%      -0.02    -0.04     -0.08
75%       0.84     0.66      0.75
max       1.76     3.68      3.24


In [16]:
# Side-by-side comparison
print("BEFORE vs AFTER Scaling (first 5 rows):")
print("\nBEFORE:")
print(X[['age', 'bmi', 'children']].head())
print("\nAFTER (StandardScaler):")
print(X_scaled[['age', 'bmi', 'children']].head().round(3))

BEFORE vs AFTER Scaling (first 5 rows):

BEFORE:
   age     bmi  children
0   19  27.900         0
1   18  33.770         1
2   28  33.000         3
3   33  22.705         0
4   32  28.880         0

AFTER (StandardScaler):
     age    bmi  children
0 -1.440 -0.453    -0.909
1 -1.512  0.509    -0.079
2 -0.799  0.383     1.580
3 -0.443 -1.305    -0.909
4 -0.514 -0.292    -0.909


### Let's Also See MinMaxScaler in Action

For comparison, here's how MinMaxScaler transforms the same data:

In [17]:
# MinMaxScaler demonstration - scales to 0-1 range
minmax_scaler = MinMaxScaler()

X_minmax = X.copy()
X_minmax[cols_to_scale] = minmax_scaler.fit_transform(X[cols_to_scale])

print("COMPARISON: StandardScaler vs MinMaxScaler")
print("=" * 60)

print("\nOriginal values (first 5 rows):")
print(X[['age', 'bmi', 'children']].head())

print("\n" + "-" * 60)
print("\nStandardScaler (mean=0, std=1):")
print("Values range: roughly -3 to +3")
print(X_scaled[['age', 'bmi', 'children']].head().round(3))

print("\n" + "-" * 60)
print("\nMinMaxScaler (min=0, max=1):")
print("Values range: exactly 0 to 1")
print(X_minmax[['age', 'bmi', 'children']].head().round(3))

COMPARISON: StandardScaler vs MinMaxScaler

Original values (first 5 rows):
   age     bmi  children
0   19  27.900         0
1   18  33.770         1
2   28  33.000         3
3   33  22.705         0
4   32  28.880         0

------------------------------------------------------------

StandardScaler (mean=0, std=1):
Values range: roughly -3 to +3
     age    bmi  children
0 -1.440 -0.453    -0.909
1 -1.512  0.509    -0.079
2 -0.799  0.383     1.580
3 -0.443 -1.305    -0.909
4 -0.514 -0.292    -0.909

------------------------------------------------------------

MinMaxScaler (min=0, max=1):
Values range: exactly 0 to 1
     age    bmi  children
0  0.022  0.321       0.0
1  0.000  0.479       0.2
2  0.217  0.458       0.6
3  0.326  0.181       0.0
4  0.304  0.348       0.0


---

## 1.5 What ML-Ready Data Looks Like

### Before we move on - let's be VERY clear about what we need!

After completing Steps 1-3, your data should look like this:

In [18]:
# Let's look at our ML-ready features
print("Our ML-Ready Features (X_scaled):")
print("="*60)
X_scaled.head(10)

Our ML-Ready Features (X_scaled):


,age,sex,bmi,children,smoker,region_northeast,region_northwest,region_southeast,region_southwest
0,-1.440418,0,-0.453160,-0.909234,1,False,False,False,True
1,-1.511647,1,0.509422,-0.079442,0,False,False,True,False
2,-0.799350,1,0.383155,1.580143,0,False,False,True,False
3,-0.443201,1,-1.305052,-0.909234,0,False,True,False,False
4,-0.514431,1,-0.292456,-0.909234,0,False,True,False,False
5,-0.585661,0,-0.807363,-0.909234,0,False,False,True,False
6,0.482785,0,0.455307,-0.079442,0,False,False,True,False
7,-0.158282,0,-0.479397,1.580143,0,False,True,False,False
8,-0.158282,1,-0.136672,0.750351,0,True,False,False,False
9,1.480002,0,-0.790965,-0.909234,0,False,True,False,False


### The ML-Ready Data Checklist

Your data is ready for machine learning when:

| Requirement | What to Check | Our Data |
|-------------|---------------|----------|
| **All Numeric** | No text, no "yes"/"no", no categories | ✓ |
| **No Special Characters** | No "$", "%", "N/A", or other symbols | ✓ |
| **Scaled Values** | Numbers roughly between -3 and +3 (or 0 and 1) | ✓ |
| **No Missing Values** | No NaN or null values | ✓ |
| **No ID Columns** | Row identifiers removed | ✓ |

In [19]:
# Let's verify each requirement:

print("ML-READY DATA VALIDATION")
print("=" * 60)

# 1. All numeric?
all_numeric = X_scaled.select_dtypes(include=['number']).shape[1] == X_scaled.shape[1]
print(f"\n1. All columns numeric? {'' if all_numeric else ''}")
print(f"   Data types: {X_scaled.dtypes.unique().tolist()}")

# 2. No missing values?
no_missing = X_scaled.isnull().sum().sum() == 0
print(f"\n2. No missing values? {'' if no_missing else ''}")
print(f"   Total missing: {X_scaled.isnull().sum().sum()}")

# 3. Scaled values? (check mean near 0 for scaled columns)
print(f"\n3. Scaled appropriately? ")
print(f"   age - mean: {X_scaled['age'].mean():.4f}, std: {X_scaled['age'].std():.4f}")
print(f"   bmi - mean: {X_scaled['bmi'].mean():.4f}, std: {X_scaled['bmi'].std():.4f}")

# 4. Value ranges
print(f"\n4. Value ranges look reasonable?")
print(f"   Min value in dataset: {X_scaled.min().min():.3f}")
print(f"   Max value in dataset: {X_scaled.max().max():.3f}")

print("\n" + "=" * 60)
print(" DATA IS ML-READY!")

ML-READY DATA VALIDATION

1. All columns numeric? 
   Data types: [dtype('float64'), dtype('int64'), dtype('bool')]

2. No missing values? 
   Total missing: 0

3. Scaled appropriately? 
   age - mean: -0.0000, std: 1.0004
   bmi - mean: -0.0000, std: 1.0004

4. Value ranges look reasonable?
   Min value in dataset: -2.411
   Max value in dataset: 3.684

 DATA IS ML-READY!


### Visual: Before vs After the Full Process

In [20]:
# Compare original data vs ML-ready data
print("ORIGINAL DATA (not ML-ready):")
print("-" * 50)
print(df.head(3).to_string())
print("\n Problems: text values ('male', 'yes', 'southwest'), unscaled numbers")

print("\n")
print("ML-READY DATA:")
print("-" * 50)
print(X_scaled.head(3).round(3).to_string())
print("\n All numbers, scaled values, no text!")

ORIGINAL DATA (not ML-ready):
--------------------------------------------------
   age     sex    bmi  children smoker     region     charges
0   19  female  27.90         0    yes  southwest  16884.9240
1   18    male  33.77         1     no  southeast   1725.5523
2   28    male  33.00         3     no  southeast   4449.4620

 Problems: text values ('male', 'yes', 'southwest'), unscaled numbers


ML-READY DATA:
--------------------------------------------------
     age  sex    bmi  children  smoker  region_northeast  region_northwest  region_southeast  region_southwest
0 -1.440    0 -0.453    -0.909       1             False             False             False              True
1 -1.512    1  0.509    -0.079       0             False             False              True             False
2 -0.799    1  0.383     1.580       0             False             False              True             False

 All numbers, scaled values, no text!


---

## 1.6 Step 4: Train/Test Split

The final step! We need to split our data into:
- **Training set (~80%)**: Used to train the model
- **Test set (~20%)**: Held back to evaluate the model

**Why split?** We need unseen data to test if our model actually learned patterns or just memorized the training data.

In [21]:
# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,        # Features
    y,               # Target
    test_size=0.2,   # 20% for testing
    random_state=42  # For reproducibility
)

print("Train/Test Split Complete!")
print("=" * 40)
print(f"Training features: {X_train.shape}")
print(f"Testing features:  {X_test.shape}")
print(f"Training target:   {y_train.shape}")
print(f"Testing target:    {y_test.shape}")

Train/Test Split Complete!
Training features: (1069, 9)
Testing features:  (268, 9)
Training target:   (1069,)
Testing target:    (268,)


In [22]:
# Verify the split percentages
total = len(X_scaled)
train_pct = len(X_train) / total * 100
test_pct = len(X_test) / total * 100

print(f"Training: {len(X_train)} rows ({train_pct:.0f}%)")
print(f"Testing:  {len(X_test)} rows ({test_pct:.0f}%)")

Training: 1069 rows (80%)
Testing:  268 rows (20%)


---

## 1.7 Summary: The Complete ML Data Prep Pipeline

```
Raw Data
    |
    v
[1. DROP] Remove ID columns, irrelevant features
    |
    v
[2. ENCODE] Convert categories to numbers
    |   - LabelEncoder for binary (yes/no, male/female)
    |   - One-Hot for multi-category (region, work_type)
    v
[3. SCALE] Normalize numerical features
    |   - StandardScaler: mean=0, std=1 (default choice)
    |   - MinMaxScaler: range 0-1 (when needed)
    v
[4. SPLIT] Divide into train/test sets
    |   - 80% training, 20% testing
    v
ML-Ready Data!
```

**Your data is ready when:**
- Every value is a number (no text, no symbols)
- Scaled values (small decimals between -3 and +3, or 0 and 1)
- No missing values (no NaN, no "N/A")
- No ID columns

---

---

# PART 2: Your Turn - Stroke Dataset

Now apply the same 4-step process to the stroke prediction dataset!

**Remember from EDA:**
- This dataset has a `bmi` column with "N/A" text values
- It has an `id` column that should be dropped
- More categorical columns to encode

---

## 2.1 Setup - Load the Data

In [23]:
# Load the stroke dataset
stroke_df = pd.read_csv('stroke_prediction.csv')

print(f"Dataset shape: {stroke_df.shape}")
stroke_df.head()

Dataset shape: (5110, 12)


,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


In [25]:
# Check data types and identify what needs work
print("Data types:")
print(stroke_df.dtypes)
print("\n" + "="*50)
print("\nCategorical columns (need encoding):")
print(stroke_df.select_dtypes(include=['object']).columns.tolist())

Data types:
id                     int64
gender                object
age                  float64
hypertension           int64
heart_disease          int64
ever_married          object
work_type             object
Residence_type        object
avg_glucose_level    float64
bmi                  float64
smoking_status        object
stroke                 int64
dtype: object


Categorical columns (need encoding):
['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']


## 2.2 Quick Data Cleaning (From EDA)

Handle the "N/A" values in the `bmi` column before we proceed.

In [26]:
# Check for 'N/A' strings in bmi
print(f"Rows with 'N/A' in bmi: {(stroke_df['bmi'] == 'N/A').sum()}")

# YOUR CODE: Handle the N/A values
# Option 1: Drop these rows
# stroke_df = stroke_df[stroke_df['bmi'] != 'N/A']

# Option 2: Replace with median (need to convert to numeric first)
# stroke_df['bmi'] = pd.to_numeric(stroke_df['bmi'], errors='coerce')
# stroke_df['bmi'] = stroke_df['bmi'].fillna(stroke_df['bmi'].median())



Rows with 'N/A' in bmi: 0


---

## 2.3 Step 1: Drop Unnecessary Columns

**Task:** Drop the `id` column - it's just a row identifier with no predictive value.

In [ ]:
# YOUR CODE: Drop the 'id' column


# Verify it's gone
print("Columns after dropping 'id':")
print(stroke_df.columns.tolist())

---

## 2.4 Step 2: Encode Categorical Variables

**Your categorical columns:**

| Column | Values | Encoding Method |
|--------|--------|----------------|
| gender | Male, Female, Other | Label or One-Hot |
| ever_married | Yes, No | Label Encoding |
| work_type | 5 categories | One-Hot Encoding |
| Residence_type | Urban, Rural | Label Encoding |
| smoking_status | 4 categories | One-Hot Encoding |

In [ ]:
# First, see the unique values in each categorical column
cat_cols = ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']

for col in cat_cols:
    print(f"{col}: {stroke_df[col].unique()}")

In [ ]:
# YOUR CODE: Create a copy and apply Label Encoding to binary columns
stroke_encoded = stroke_df.copy()
le = LabelEncoder()

# Encode ever_married (Yes/No)


# Encode Residence_type (Urban/Rural)


# Encode gender (you can use Label or One-Hot, your choice)



In [ ]:
# YOUR CODE: Apply One-Hot Encoding to multi-category columns
# work_type and smoking_status



In [ ]:
# Verify all columns are now numeric
print("Data types after encoding:")
print(stroke_encoded.dtypes)

---

## 2.5 Step 3: Scale Numerical Features

**Numerical columns to scale:**
- `age`
- `avg_glucose_level`
- `bmi`

In [ ]:
# YOUR CODE: Separate features (X) from target (y)
# Target is 'stroke' column

X = 
y = 

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

In [ ]:
# YOUR CODE: Apply StandardScaler to numerical columns
scaler = StandardScaler()
cols_to_scale = ['age', 'avg_glucose_level', 'bmi']

X_scaled = X.copy()

# Scale the columns



In [ ]:
# Verify scaling worked
print("After scaling - check means are ~0 and std ~1:")
print(X_scaled[cols_to_scale].describe().round(2))

---

## 2.6 Validate Your ML-Ready Data

In [ ]:
# Run the validation checklist
print("ML-READY DATA VALIDATION")
print("=" * 60)

# 1. All numeric?
all_numeric = X_scaled.select_dtypes(include=['number']).shape[1] == X_scaled.shape[1]
print(f"\n1. All columns numeric? {'' if all_numeric else ''}")

# 2. No missing values?
no_missing = X_scaled.isnull().sum().sum() == 0
print(f"2. No missing values? {'' if no_missing else ''} ({X_scaled.isnull().sum().sum()} missing)")

# 3. Value ranges
print(f"3. Value ranges:")
print(f"   Min: {X_scaled.min().min():.3f}")
print(f"   Max: {X_scaled.max().max():.3f}")

# 4. Final shape
print(f"\n4. Final shape: {X_scaled.shape}")

if all_numeric and no_missing:
    print("\n" + "=" * 60)
    print(" DATA IS ML-READY!")
else:
    print("\n Fix the issues above before proceeding!")

In [ ]:
# Look at your final ML-ready data
print("Your ML-Ready Features:")
X_scaled.head()

---

## 2.7 Step 4: Train/Test Split

In [ ]:
# YOUR CODE: Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    # Fill in the parameters
)

print("Train/Test Split Complete!")
print("=" * 40)
print(f"Training features: {X_train.shape}")
print(f"Testing features:  {X_test.shape}")
print(f"Training target:   {y_train.shape}")
print(f"Testing target:    {y_test.shape}")

---

# Congratulations!

You've completed the full ML data preparation pipeline:

1. **Dropped** unnecessary columns (id)
2. **Encoded** categorical variables (text → numbers)
3. **Scaled** numerical features (standardized to mean=0, std=1)
4. **Split** into training and testing sets

**Your data is now ready to feed into machine learning models!**

---

## Key Takeaways

| Concept | What It Means | When to Use |
|---------|---------------|-------------|
| **LabelEncoder** | Converts text categories to 0, 1, 2... | Binary categories (yes/no) |
| **One-Hot Encoding** | Creates new columns for each category | 3+ categories with no order |
| **StandardScaler** | Transforms to mean=0, std=1 | Most ML models (default choice) |
| **MinMaxScaler** | Transforms to range 0-1 | When you need bounded values |
| **train_test_split** | Divides data for training and evaluation | Always before modeling |

### Scaler Quick Reference:
- **StandardScaler** → Use by default, handles outliers better
- **MinMaxScaler** → Use when you need values exactly between 0 and 1

**Next class:** We'll actually build ML models using this prepared data!

---